# Analyzing data

In [ ]:
import dol 

s = dol.JsonFiles('/Users/thorwhalen/Dropbox/_odata/finance/mood/news/searches')
len(s)

52067

In [ ]:
from pprint import pprint

headlines_kinds = ["yahoo_finance_headlines", "newsdata", "yahoo_finance"]

def headline_store(store, headline_kind):
    return dol.cache_iter(dol.filt_iter(store, filt=lambda k: k.startswith(headline_kind + '/')))

def peek_at_data(store, headline_kind):
    headlines_store = headline_store(store, headline_kind)
    k, v = next(iter(headlines_store.items()))
    print(f"Key: {k}")
    print("Value:")
    pprint(v)

In [ ]:
# Conversion helpers and per-source DataFrame builders
import re
from datetime import datetime, timezone
import pandas as pd
import dol

# --- generic extractors ---

def _first(d: dict, keys):
    for k in keys:
        if k in d and d[k] not in (None, ""):
            return d[k]
    return None


def _parse_timestamp(val):
    if val is None:
        return None
    # epoch seconds or ms
    try:
        if isinstance(val, (int, float)):
            v = float(val)
            if v > 1e12:  # ms -> s
                v = v / 1000.0
            return datetime.fromtimestamp(v, tz=timezone.utc)
    except Exception:
        pass
    # strings
    if isinstance(val, str):
        s = val.strip()
        # ISO-8601
        try:
            return datetime.fromisoformat(s.replace("Z", "+00:00"))
        except Exception:
            pass
        # RFC 2822 and common news formats
        fmts = (
            "%a, %d %b %Y %H:%M:%S %z",
            "%a, %d %b %Y %H:%M:%S %Z",
            "%Y-%m-%d %H:%M:%S",
            "%Y-%m-%d",
        )
        for fmt in fmts:
            try:
                return datetime.strptime(s, fmt)
            except Exception:
                continue
    return None


def _extract_row(item: dict):
    # Pick timestamp from common fields
    ts_field_val = _first(
        item,
        [
            "timestamp",
            "providerPublishTime",
            "pubDate",
            "published_at",
            "publishedAt",
            "date",
            "time",
            "created_at",
            "updated_at",
        ],
    )
    ts = _parse_timestamp(ts_field_val)
    if ts is None and isinstance(ts_field_val, str):
        # Sometimes numeric string epoch
        try:
            ts = _parse_timestamp(float(ts_field_val))
        except Exception:
            pass

    # Choose best headline text
    text = _first(item, ["title", "headline", "text", "name"]) or _first(
        item, ["summary", "description"]
    )

    row = {
        "timestamp": ts,
        "text": text,
    }

    # A few useful, commonly present extras if available
    for fld in (
        "id",
        "uuid",
        "url",
        "link",
        "publisher",
        "source",
        "source_id",
        "author",
        "category",
        "tickers",
        "symbols",
        "type",
        "language",
        "sentiment",
    ):
        if fld in item:
            row[fld] = item[fld]

    # Keep the full original payload so no information is lost
    row["raw"] = item
    return row


def _iter_rows_from_value(v):
    """Yield normalized rows from a value which can be dict/list/single-item."""
    items = None
    if isinstance(v, dict):
        # Look for typical containers
        for key in ("items", "articles", "news", "result", "results", "stories", "data", "feed"):
            if key in v:
                maybe = v[key]
                if isinstance(maybe, list) and (not maybe or isinstance(maybe[0], dict)):
                    items = maybe
                    break
                if isinstance(maybe, dict):
                    # nested dict may again hold lists
                    for k2 in ("items", "articles", "news", "result", "results", "stories", "feed"):
                        if k2 in maybe and isinstance(maybe[k2], list):
                            items = maybe[k2]
                            break
                    if items is not None:
                        break
        # If still not found, maybe v itself is an item
        if items is None and any(k in v for k in ("title", "headline", "text")):
            items = [v]
        # Or find first list of dicts
        if items is None:
            for _k, _v in v.items():
                if isinstance(_v, list) and _v and isinstance(_v[0], dict):
                    items = _v
                    break
    elif isinstance(v, list):
        items = v

    if not items:
        return

    for item in items:
        if isinstance(item, dict):
            yield _extract_row(item)


def _to_df_by_kind(store, kind: str) -> pd.DataFrame:
    rows = []
    hs = headline_store(store, kind)
    for k, v in hs.items():
        for row in _iter_rows_from_value(v):
            row.setdefault("source_kind", kind)
            row.setdefault("store_key", k)
            # Optional: try to parse a date from the key if timestamp missing
            if row.get("timestamp") is None:
                m = re.search(r"(20\d{2}-\d{2}-\d{2})", str(k))
                if m:
                    try:
                        row["timestamp"] = datetime.fromisoformat(m.group(1)).replace(tzinfo=timezone.utc)
                    except Exception:
                        pass
            rows.append(row)
    df = pd.DataFrame(rows)
    # Ensure required columns exist
    for col in ("timestamp", "text"):
        if col not in df.columns:
            df[col] = None
    return df


# --- Public API: three conversion functions ---

def to_df_yahoo_finance_headlines(store) -> pd.DataFrame:
    return _to_df_by_kind(store, "yahoo_finance_headlines")


def to_df_newsdata(store) -> pd.DataFrame:
    return _to_df_by_kind(store, "newsdata")


def to_df_yahoo_finance(store) -> pd.DataFrame:
    return _to_df_by_kind(store, "yahoo_finance")


In [ ]:
headline_kind = headlines_kinds[0]
print(f"{headline_kind=}")
peek_at_data(s, headline_kind)

headline_kind='yahoo_finance_headlines'
Key: yahoo_finance_headlines/2025-07-22/2025-07-22--18-00-03__.json
Value:
['3 No-Brainer High-Yield Stocks to Buy With $500 Right Now',
 'Stellantis Takes $350 Million Hit From Tariffs',
 'Global Markets Mixed as Attention Turns to Corporate Earnings',
 "'Earnings misses are going to get punished more than usual': Wall Street "
 'raises the stakes as stocks hit records',
 'Mark Tilbury’s 5 steps for financial freedom',
 'Market Update: ABT, DPZ, GE, KEY, SJM, VZ',
 'Why Rocket Lab Stock Skyrocketed Last Week',
 'High-yield savings rates today: July 21, 2025 | Best APYs exceed 4% ahead of '
 'next week’s Fed meeting',
 'Galp Gets Offers for Mopane Discovery in Namibia',
 'Stock Futures Rise as Investors Weigh Up Earnings and Trade']


In [ ]:
# Preview a few rows for each kind
from IPython.display import display
kinds = ["yahoo_finance_headlines", "newsdata", "yahoo_finance"]
funcs = {
    "yahoo_finance_headlines": to_df_yahoo_finance_headlines,
    "newsdata": to_df_newsdata,
    "yahoo_finance": to_df_yahoo_finance,
}

for k in kinds:
    df = funcs[k](s)
    print(f"{k}: shape={df.shape}")
    cols = [c for c in ["timestamp", "text", "publisher", "url", "store_key"] if c in df.columns]
    display(df[cols].head(3))

yahoo_finance_headlines: shape=(0, 2)


,timestamp,text


newsdata: shape=(254402, 10)


,timestamp,text,store_key
0,2025-07-22 00:01:13,"GOP to America: Forget the Epstein files, we’r...",newsdata/2025-07-22/2025-07-22--12-02-09__Rece...
1,2025-07-21 23:53:25,"Thune stuck between Trump's demands, members' ...",newsdata/2025-07-22/2025-07-22--12-02-09__Rece...
2,2025-07-21 23:50:53,Senate GOP faces tough call over Trump's deman...,newsdata/2025-07-22/2025-07-22--12-02-09__Rece...


yahoo_finance: shape=(209513, 9)


,timestamp,text,publisher,store_key
0,2025-07-22 11:57:05+00:00,UK hotel tax plan scrapped after Treasury push...,Hotel Management Network,yahoo_finance/2025-07-22/2025-07-22--12-00-06_...
1,2025-07-22 11:56:29+00:00,Community Financial: Q2 Earnings Snapshot,Associated Press Finance,yahoo_finance/2025-07-22/2025-07-22--12-00-06_...
2,2025-07-22 11:55:03+00:00,Apple supplier Goertek to buy Luen Fung Commer...,Reuters,yahoo_finance/2025-07-22/2025-07-22--12-00-06_...


In [ ]:
headline_kind = headlines_kinds[1]
print(f"{headline_kind=}")
peek_at_data(s, headline_kind)

headline_kind='newsdata'
Key: newsdata/2025-07-22/2025-07-22--12-02-09__Recession.json
Value:
[{'ai_content': 'ONLY AVAILABLE IN PROFESSIONAL AND CORPORATE PLANS',
  'ai_org': 'ONLY AVAILABLE IN CORPORATE PLANS',
  'ai_region': 'ONLY AVAILABLE IN CORPORATE PLANS',
  'ai_summary': 'ONLY AVAILABLE IN PAID PLANS',
  'ai_tag': 'ONLY AVAILABLE IN PROFESSIONAL AND CORPORATE PLANS',
  'article_id': 'e182b5359157f1947b42f27625393291',
  'category': ['politics'],
  'content': 'ONLY AVAILABLE IN PAID PLANS',
  'country': ['united states of america'],
  'creator': ['rss@dailykos.com (Emily Singer)'],
  'description': 'House Republicans are still trying to bury the Epstein '
                 'scandal, hopeful that Americans will simply forget about it '
                 'during the chamber’s summer break.According to Politico, the '
                 'GOP-controlled House will not take any votes on the Epstein '
                 'files before the chamber begins its month-long recess in '
          

In [ ]:
headline_kind = headlines_kinds[2]
print(f"{headline_kind=}")
peek_at_data(s, headline_kind)

headline_kind='yahoo_finance'
Key: yahoo_finance/2025-07-22/2025-07-22--12-00-06__Cyberattack.json
Value:
{'count': 10,
 'explains': [],
 'lists': [],
 'nav': [],
 'news': [{'link': 'https://finance.yahoo.com/m/c65a0cdf-fca4-3ed1-bec7-278362290680/uk-hotel-tax-plan-scrapped.html',
           'providerPublishTime': 1753185425,
           'publisher': 'Hotel Management Network',
           'thumbnail': {'resolutions': [{'height': 533,
                                          'tag': 'original',
                                          'url': 'https://s.yimg.com/uu/api/res/1.2/N6YsB5CPsG3pwc5vYDhCFA--~B/aD01MzM7dz0xMDAwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/hotel_management_network_805/8fcd1175e01ec3f31558233b9e4adcf9',
                                          'width': 1000},
                                         {'height': 140,
                                          'tag': '140x140',
                                          'url': 'https://s.yimg.com/uu/api/res/1.2/qSBF

## More

In [ ]:
import pandas as pd 

headlines_kind = 'yahoo_finance'  # yahoo_finance_headlines, newsdata, yahoo_finance
headlines_store = dol.JsonFiles(f'/Users/thorwhalen/Dropbox/_odata/finance/mood/news/searches/{headlines_kind}')
headlines = pd.DataFrame(headlines_store.values())
print(f"{headlines.shape=}")
headlines.iloc[0]

headlines.shape=(24476, 19)


explains                                                                         []
count                                                                            10
quotes                                                                           []
news                              [{'uuid': 'c65a0cdf-fca4-3ed1-bec7-27836229068...
nav                                                                              []
lists                                                                            []
researchReports                                                                  []
screenerFieldResults                                                             []
totalTime                                                                        48
timeTakenForQuotes                                                              430
timeTakenForNews                                                                600
timeTakenForAlgowatchlist                                                   

In [ ]:
len(headlines_store)

24476

In [ ]:
k, v = next(iter(headlines_store.items()))
print(k)
v

2025-07-22/2025-07-22--12-00-06__Cyberattack.json


{'explains': [],
 'count': 10,
 'quotes': [],
 'news': [{'uuid': 'c65a0cdf-fca4-3ed1-bec7-278362290680',
   'title': 'UK hotel tax plan scrapped after Treasury pushback',
   'publisher': 'Hotel Management Network',
   'link': 'https://finance.yahoo.com/m/c65a0cdf-fca4-3ed1-bec7-278362290680/uk-hotel-tax-plan-scrapped.html',
   'providerPublishTime': 1753185425,
   'type': 'STORY',
   'thumbnail': {'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/N6YsB5CPsG3pwc5vYDhCFA--~B/aD01MzM7dz0xMDAwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/hotel_management_network_805/8fcd1175e01ec3f31558233b9e4adcf9',
      'width': 1000,
      'height': 533,
      'tag': 'original'},
     {'url': 'https://s.yimg.com/uu/api/res/1.2/qSBFhr5691FMaUUY2jtkRA--~B/Zmk9ZmlsbDtoPTE0MDtweW9mZj0wO3c9MTQwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/hotel_management_network_805/8fcd1175e01ec3f31558233b9e4adcf9',
      'width': 140,
      'height': 140,
      'tag': '140x140'}]}},
  {'uuid': '95d0b7d9-efba

In [ ]:
newsdata_store = dol.JsonFiles('/Users/thorwhalen/Dropbox/_odata/finance/mood/news/searches/newsdata')
newsdata = pd.DataFrame(newsdata_store.values())
print(f"{newsdata.shape=}")
newsdata.iloc[0]

newsdata.shape=(96, 0)


Series([], Name: 0, dtype: float64)

In [ ]:
list(newsdata_store)

['2025-03-12/2025-03-12--12-00-20____data_financial_news_search_words_json.json',
 '2025-03-12/2025-03-12--06-00-17____data_financial_news_search_words_json.json',
 '2025-03-12/2025-03-12--00-00-38____data_financial_news_search_words_json.json',
 '2025-02-25/2025-02-25--18-00-22____data_inancial_news_search_words_json.json',
 '2025-02-25/2025-02-25--00-00-50____data_inancial_news_search_words_json.json',
 '2025-02-25/2025-02-25--12-00-23____data_inancial_news_search_words_json.json',
 '2025-02-25/2025-02-25--06-00-19____data_inancial_news_search_words_json.json',
 '2025-02-22/2025-02-22--18-00-18____data_inancial_news_search_words_json.json',
 '2025-02-22/2025-02-22--06-00-22____data_inancial_news_search_words_json.json',
 '2025-02-22/2025-02-22--12-00-23____data_inancial_news_search_words_json.json',
 '2025-02-22/2025-02-22--00-00-49____data_inancial_news_search_words_json.json',
 '2025-02-23/2025-02-23--18-00-20____data_inancial_news_search_words_json.json',
 '2025-02-23/2025-02-23--

In [ ]:
from mood.tools import search_and_save_news, DFLT_STORE; t = search_and_save_news('APPL', source='newsdata'); print(DFLT_STORE); print(t);

In [ ]:
s = dol.Files(DFLT_STORE)
list(s)

['yahoo_finance_headlines/2025-02-19/2025-02-19--09-52-32__.json',
 'newsdata/2025-03-12/2025-03-12--18-26-25__APPL.json']

In [ ]:
DFLT_STORE

'/Users/thorwhalen/.config/mood/news/searches'

'/Users/thorwhalen/Dropbox/py/proj/misc/infinvest/mood/misc'

# Updating the local data with server data

In [ ]:
from xdol import update_with_policy
from xdol.updating import KeyDecision


def today_utc_date_str():
    import datetime

    today = datetime.datetime.now().date()
    return str(today)


def update_missing_or_today(target, source, *, keys_to_consider=None, verbose=True):
    """
    Update target with source, but only for keys that are missing in target or
    that contain today's date.
    """
    today = today_utc_date_str()

    def _missing_or_today_decider(key, target_info, source_info) -> KeyDecision:
        if source_info is None:
            return "SKIP"
        elif target_info is None or today in str(key):
            return "COPY"

        return "SKIP"

    return update_with_policy(
        target,
        source,
        policy=_missing_or_today_decider,
        keys_to_consider=keys_to_consider,
        verbose=verbose,
    )

In [ ]:
from sshdol import SshFiles
import dol 

src_rootdir = '/root/.config/mood/news/searches'
src = SshFiles(
    'thorwhalen', rootdir=src_rootdir, max_levels=None, include_directories=False
)

target_rootdir = '/Users/thorwhalen/Dropbox/_odata/finance/mood/news/searches'
target = dol.mk_dirs_if_missing(dol.Files(target_rootdir))

print(f"Remote (src): {len(src)} files in {src.rootdir}")
print(f"Local (target): {len(target)} files in {target.rootdir}")

Remote (src): 51970 files in /root/.config/mood/news/searches
Local (target): 52067 files in /Users/thorwhalen/Dropbox/_odata/finance/mood/news/searches/


In [ ]:
# Sync
src.sync_to(target.rootdir) 

In [ ]:
print(f"Local (target): {len(target)} files after remote->local sync")

Local (target): 52067 files after remote->local sync
